# Demo 2: Biomarker Synergy in Disease Classification

**Research Question**: Which biomarker combinations best distinguish PD progression stages?

**Hypothesis**: Combining CSF biomarkers (alpha-synuclein, tau) with plasma markers (ptau217) and clinical symptoms will reveal synergistic patterns that better identify progression risk than individual biomarkers alone.

**Multi-Modal Integration**:
- CSF: alpha-synuclein, abeta, ptau, tau
- Plasma: ptau217_plasma, bd_tau_plasma
- Clinical: UPDRS scores, disease duration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Add src to path
sys.path.insert(0, str(Path('../..').resolve()))

from src.data_loader import load_ppmi_data, filter_by_cohort
from src.data_preprocessing import extract_feature_groups

# Set display options
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['figure.dpi'] = 100

## 1. Load Data and Filter PD Cohort

In [ ]:
# Load PPMI data
df = load_ppmi_data()
print(f"Total dataset: {df.shape}")

# Focus on PD and Healthy Controls for comparison
df_filtered = filter_by_cohort(df, ['PD', 'HC'])
print(f"PD + HC cohorts: {df_filtered.shape}")
print(f"\nCohort distribution:\n{df_filtered['COHORT'].value_counts()}")

## 2. Identify Biomarker Features

In [ ]:
# Extract biomarker features
biomarker_cols = extract_feature_groups(df_filtered, 'biomarker')
print(f"Biomarker features ({len(biomarker_cols)}):")
for i, col in enumerate(biomarker_cols, 1):
    missing_pct = (df_filtered[col].isnull().sum() / len(df_filtered) * 100)
    print(f"  {i:2d}. {col:30s} - Missing: {missing_pct:5.1f}%")

## 3. CSF Biomarker Analysis

In [ ]:
# Focus on key CSF biomarkers
csf_markers = [col for col in biomarker_cols if 'csf' in col.lower() or 
               any(marker in col.lower() for marker in ['synuclein', 'abeta', 'tau'])]

print(f"CSF biomarkers: {csf_markers}")

# Create subset with non-null CSF data
df_csf = df_filtered.dropna(subset=csf_markers, how='all')
print(f"\nSamples with CSF data: {len(df_csf)}")

In [ ]:
# Visualize CSF biomarker distributions by cohort
n_markers = len(csf_markers)
if n_markers > 0:
    fig, axes = plt.subplots((n_markers + 1) // 2, 2, figsize=(14, 4 * ((n_markers + 1) // 2)))
    axes = axes.flatten() if n_markers > 1 else [axes]
    
    for i, marker in enumerate(csf_markers):
        if i < len(axes):
            df_csf.boxplot(column=marker, by='COHORT', ax=axes[i])
            axes[i].set_title(f'{marker} by Cohort')
            axes[i].set_xlabel('Cohort')
            axes[i].set_ylabel(marker)
            plt.sca(axes[i])
            plt.xticks(rotation=0)
    
    # Hide extra subplots
    for i in range(n_markers, len(axes)):
        axes[i].set_visible(False)
    
    plt.suptitle('CSF Biomarker Distributions by Cohort', fontsize=16, y=1.0)
    plt.tight_layout()
    plt.show()

## 4. Plasma Biomarker Analysis (March 2025 Addition)

In [ ]:
# Focus on plasma biomarkers
plasma_markers = [col for col in biomarker_cols if 'plasma' in col.lower()]

print(f"Plasma biomarkers: {plasma_markers}")

if plasma_markers:
    df_plasma = df_filtered.dropna(subset=plasma_markers, how='all')
    print(f"\nSamples with plasma data: {len(df_plasma)}")
    
    # Summary statistics
    print("\nPlasma biomarker summary:")
    print(df_plasma[plasma_markers].describe())

In [ ]:
# Visualize plasma biomarkers by cohort
if plasma_markers:
    fig, axes = plt.subplots(1, len(plasma_markers), figsize=(6 * len(plasma_markers), 5))
    if len(plasma_markers) == 1:
        axes = [axes]
    
    for i, marker in enumerate(plasma_markers):
        df_plasma.boxplot(column=marker, by='COHORT', ax=axes[i])
        axes[i].set_title(f'{marker} by Cohort')
        axes[i].set_xlabel('Cohort')
        axes[i].set_ylabel(marker)
    
    plt.suptitle('Plasma Biomarker Distributions by Cohort', fontsize=16)
    plt.tight_layout()
    plt.show()

## 5. Biomarker Correlation Analysis

In [ ]:
# Combine CSF and plasma biomarkers
all_biomarkers = csf_markers + plasma_markers

if all_biomarkers:
    # Calculate correlation matrix
    biomarker_data = df_filtered[all_biomarkers]
    corr_matrix = biomarker_data.corr()
    
    # Visualize correlation heatmap
    plt.figure(figsize=(12, 10))
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
                center=0, square=True, linewidths=1,
                cbar_kws={"shrink": 0.8})
    plt.title('Biomarker Correlation Matrix\n(CSF + Plasma)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 6. Multi-Modal Integration: Biomarkers + Clinical

Combine biomarkers with clinical features to identify synergistic patterns

In [ ]:
# Extract UPDRS features
updrs_cols = extract_feature_groups(df_filtered, 'updrs')
print(f"UPDRS features: {updrs_cols[:5]}...")  # Show first 5

# Create multi-modal feature set
multimodal_features = all_biomarkers + updrs_cols[:5]  # Use top 5 UPDRS

# Filter samples with both biomarker and clinical data
df_multimodal = df_filtered.dropna(subset=multimodal_features, thresh=len(multimodal_features)//2)
print(f"\nSamples with multi-modal data: {len(df_multimodal)}")
print(f"PD samples: {(df_multimodal['COHORT'] == 'PD').sum()}")
print(f"HC samples: {(df_multimodal['COHORT'] == 'HC').sum()}")

## 7. Feature Importance via Random Forest

Identify which biomarker combinations are most discriminative

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

if len(df_multimodal) > 50:  # Minimum sample requirement
    # Prepare data
    X = df_multimodal[multimodal_features].fillna(df_multimodal[multimodal_features].median())
    y = LabelEncoder().fit_transform(df_multimodal['COHORT'])
    
    # Train Random Forest
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X, y)
    
    # Get feature importances
    importances = pd.DataFrame({
        'Feature': multimodal_features,
        'Importance': rf.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    # Visualize top 15 features
    plt.figure(figsize=(12, 6))
    top_features = importances.head(15)
    plt.barh(range(len(top_features)), top_features['Importance'], color='steelblue')
    plt.yticks(range(len(top_features)), top_features['Feature'])
    plt.xlabel('Feature Importance', fontsize=12)
    plt.title('Top 15 Biomarker + Clinical Features for PD Classification', 
              fontsize=14, fontweight='bold')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
    
    print("\nTop 10 Most Important Features:")
    print(importances.head(10).to_string(index=False))
else:
    print("Insufficient samples for feature importance analysis")

## 8. Key Findings and Insights

Summary of biomarker synergy discoveries

In [ ]:
print("="*60)
print("KEY FINDINGS: BIOMARKER SYNERGY ANALYSIS")
print("="*60)
print("\n1. CSF Biomarkers:")
print("   - Identified", len(csf_markers), "CSF markers")
print("   - Samples with CSF data:", len(df_csf) if csf_markers else 0)

print("\n2. Plasma Biomarkers (March 2025 Addition):")
print("   - Identified", len(plasma_markers), "plasma markers")
print("   - Samples with plasma data:", len(df_plasma) if plasma_markers else 0)

print("\n3. Multi-Modal Integration:")
print("   - Combined biomarkers + clinical features")
print("   - Total features:", len(multimodal_features))
print("   - Samples with complete data:", len(df_multimodal))

print("\n4. Feature Importance:")
if len(df_multimodal) > 50:
    top_biomarker = importances.iloc[0]
    print(f"   - Most important feature: {top_biomarker['Feature']}")
    print(f"   - Importance score: {top_biomarker['Importance']:.4f}")
    
    # Count biomarker vs clinical in top 10
    top10 = importances.head(10)
    biomarker_count = sum(1 for f in top10['Feature'] if f in all_biomarkers)
    clinical_count = 10 - biomarker_count
    print(f"   - Top 10: {biomarker_count} biomarker, {clinical_count} clinical")

print("\n" + "="*60)

## Next Steps

1. Investigate temporal changes in biomarker levels
2. Analyze biomarker-clinical interactions
3. Build predictive models using biomarker combinations
4. Validate findings across different visit timepoints
5. Compare with imaging and genetic data